# Suite EXP — Expressions

`Expressions.OfLiteral`, `OfOperation`, `OfVariable`, `OfLet` and `OfAny` follow the builder pattern (`Data`,
`Builder`, `Spec`, `resolve`) and have meta-schemas (`Schema`): ordinary object schemas tagged by `kind`, registered as
`Expressions.*`, through which expressions serialize, validate and compare like other objects. Builders implement
`Visitors.OfObject`, which is how `FromPlain(Expressions.Builders)` rebuilds `Data`. `Term`s write expressions with
methods. `Evaluators` computes them (suite EVL).

In [ ]:
import { Comparison as C, Expressions as E, JSON, Plain, Proxies, Schemas as S, Validators, YAML } from "../src/Framework/index.js";
import { AttributeError, KeyError, LookupError, NotImplementedError, ValueError } from "../src/Framework/Errors.js";
import { assert, Fake, get_property, raises, same, set_property, text } from "./support.js";

const lit = new E.OfLiteral.Builder().value(3n).create();
const op = new E.OfOperation.Builder().name("f").arguments(1n, (a) => a.as_literal("x")).arguments(lit).create();
const values = (operation: E.OfOperation.Data) => operation.arguments.map((a) => (a as E.OfLiteral.Data).value);

## EXP-01 · Literals hold one native value

In [ ]:
assert(lit.value === 3n && lit.validate().length === 0 && lit.schema_name() === "Expressions.OfLiteral");
assert(E.OfLiteral.resolve(3n).value === 3n && E.OfLiteral.resolve(lit) === lit);
assert(E.OfLiteral.resolve((b: E.OfLiteral.Builder) => b.value("x")).value === "x");
for (const value of [0n, 2n ** 70n, -0, NaN, "", new Uint8Array(), false]) { // every native type
  assert(E.OfLiteral.resolve(value).validate().length === 0);
}
assert(same(new E.OfLiteral.Data().validate(), ["a literal needs a value"]));
assert(same(new E.OfLiteral.Data([1n]).validate(), ["a literal must hold a native value, got list"]));

## EXP-02 · Operations have a name and ordered arguments

In [ ]:
assert(op.name === "f" && same(values(op), [1n, "x", 3n]) && op.arguments[2] === lit);
assert(op.validate().length === 0 && op.schema_name() === "Expressions.OfOperation");
const nested = E.OfOperation.resolve((o: E.OfOperation.Builder) => o.name("not").arguments((a) => a.as_operation(op)));
assert(same(nested.arguments, [op]) && E.OfOperation.resolve(op) === op);
assert(same(new E.OfOperation.Data().validate(), ["an operation needs a name"]));
assert(same(new E.OfOperation.Data("").validate(), ["an operation needs a name"]));
assert(same(new E.OfOperation.Data(3n).validate(), ["an operation's name must be a str, got int"]));
const bad = new E.OfOperation.Data("f", [new E.OfLiteral.Data(), new E.OfOperation.Data(null, [new E.OfLiteral.Data([])]), 7n]);
assert(same(bad.validate(), [
  "argument 0: a literal needs a value",
  "argument 1: an operation needs a name",
  "argument 1: argument 0: a literal must hold a native value, got list",
  "argument 2: not an expression: 7",
]));

## EXP-03 · Cycles are reported, not followed; shared arguments are not cycles

In [ ]:
const loop = new E.OfOperation.Builder().name("f").create();
new E.OfOperation.Builder(loop).arguments(loop).update();
assert(same(loop.validate(), ["argument 0: the expression contains a cycle"]));
assert(new E.OfOperation.Builder().name("g").arguments(op, op).create().validate().length === 0);

## EXP-04 · Builders: create, clone and update; `OfAny` selects a kind; Spec errors

In [ ]:
const source = new E.OfOperation.Builder().name("f").arguments(1n).create();
const copy = new E.OfOperation.Builder(source).arguments(2n).clone();
assert(same(values(copy), [1n, 2n]) && source.arguments.length === 1);
assert(new E.OfOperation.Builder(source).name("g").update() === source && source.name === "g");
const literal = new E.OfLiteral.Builder().value(3n).create();
assert(new E.OfLiteral.Builder(literal).value(4n).clone().value === 4n && literal.value === 3n);
assert(new E.OfLiteral.Builder(literal).value(5n).update() === literal && literal.value === 5n);
raises(ValueError, () => new E.OfLiteral.Builder(literal).create(), "create() is only valid without a source instance");
raises(ValueError, () => new E.OfOperation.Builder().clone(), "clone() is only valid with a source instance");
raises(ValueError, () => new E.OfLiteral.Builder().update(), "update() is only valid with a source instance");
raises(TypeError, () => (new E.OfLiteral.Builder().create as any)(1));
raises(TypeError, () => new E.OfOperation.Builder(3n as any), "expected operation data to build from, got int");
raises(TypeError, () => new E.OfLiteral.Builder("x" as any), "expected literal data to build from, got str");

assert((new E.OfAny.Builder().as_literal(1n).create() as E.OfLiteral.Data).value === 1n);
assert((E.OfAny.resolve((a: E.OfAny.Builder) => a.as_operation((o) => o.name("f"))) as E.OfOperation.Data).name === "f");
assert((E.OfAny.resolve(2n) as E.OfLiteral.Data).value === 2n && E.OfAny.resolve(op) === op);
raises(ValueError, () => new E.OfAny.Builder(literal).create(), "create() is only valid without a source instance");
raises(ValueError, () => new E.OfAny.Builder().clone(), "clone() is only valid with a source instance");
raises(ValueError, () => new E.OfAny.Builder().update(), "update() is only valid with a source instance");
raises(ValueError, () => new E.OfAny.Builder().create(), "no kind selected");
const cloned = new E.OfAny.Builder(literal).clone() as E.OfLiteral.Data;
assert(cloned !== literal && cloned.value === literal.value);
const shallow = new E.OfAny.Builder(op).clone() as E.OfOperation.Data;
assert(shallow !== op && shallow.name === op.name && same(shallow.arguments, op.arguments));
assert((new E.OfAny.Builder(literal).as_literal(1n).clone() as E.OfLiteral.Data).value === 1n && literal.value === 5n);
assert(new E.OfAny.Builder(literal).as_literal(9n).update() === literal && literal.value === 9n);
const target = new E.OfOperation.Builder().name("f").create();
assert(new E.OfAny.Builder(target).as_operation((o) => o.name("h").arguments(1n)).update() === target);
assert(target.name === "h" && same(values(target), [1n]));
raises(TypeError, () => new E.OfAny.Builder(literal).as_operation(op).update(), "update() cannot change the kind of the source expression");

raises(TypeError, () => E.OfAny.resolve(BigInt), "a class is not a Spec here, got int");
raises(TypeError, () => E.OfAny.resolve(() => undefined), "a Spec callable must return its builder, got None");
raises(TypeError, () => E.OfOperation.resolve(3n), "expected an operation or a callable taking its builder, got 3");
raises(TypeError, () => E.OfAny.resolve([1n]), "expected an expression, a native value or a callable taking its builder, got [1]");
raises(TypeError, () => E.OfLiteral.resolve(null), "expected a native value, a literal or a callable taking its builder, got None");

## EXP-05 · The meta-schemas are ordinary, registered schemas

In [ ]:
const kinds = [E.OfLiteral, E.OfOperation, E.OfVariable, E.OfLet] as const;
for (const schema of [...kinds.map((k) => k.Schema), E.Arguments, E.OfAny.Schema]) {
  assert(schema.validate().length === 0);
}
assert(same([...E.OfLiteral.Schema.properties.keys()], ["kind", "int", "float", "str", "bool", "bytes"]));
for (const kind of [E.OfOperation, E.OfVariable, E.OfLet]) assert(same([...kind.Schema.properties.keys()], ["kind", "name"]));
assert(globalThis.JSON.stringify(kinds.map((k) => [...k.Schema.adjacencies.keys()])) ===
  globalThis.JSON.stringify([["used_by"], ["arguments", "used_by"], ["used_by"], ["arguments", "used_by"]]));
assert(same(E.Arguments.links, ["parent", "argument"]) && same([...E.Arguments.properties.keys()], ["index"]));
assert(E.Arguments.uniques.length === 1 && same([...E.Arguments.uniques[0]!], ["argument"])); // the parent and index determine the argument
assert(same(E.OfAny.Schema.branches.map((b) => b.type), kinds.map((k) => k.Schema)));
assert(same(kinds.map((k) => k.Data.KIND), ["literal", "operation", "variable", "let"]));
const names = ["Expressions.OfLiteral", "Expressions.OfOperation", "Expressions.OfVariable", "Expressions.OfLet"];
assert(same(names.map((name) => Proxies.schema(name)), kinds.map((k) => k.Schema)));
assert(same(names.map((name) => E.Builders.schema(name)), kinds.map((k) => k.Schema)));
assert(same(kinds.map((k) => new k.Data().schema_name()), names));
assert(Proxies.name_of(E.Arguments) === "Expressions.Arguments" && E.Builders.name_of(E.Arguments) === "Expressions.Arguments");
raises(TypeError, () => E.Builders.schema("Expressions.Arguments"), "'Expressions.Arguments' is a relation; no relation builder is exposed");
raises(AttributeError, () => E.Builders.schema("Contact"), "no schema registered as 'Contact'");
raises(LookupError, () => E.Builders.name_of(new S.OfObject.Data()), "schema is not registered");

## EXP-06 · Round trips through Plain, JSON and YAML rebuild `Data`

In [ ]:
const expr = new E.OfOperation.Builder().name("f").arguments(
  (a) => a.as_operation((o) => o.name("eq").arguments(1n, 1.5)), "s", new Uint8Array([1]), true).create();
const expected = JSON.ToJSON.Reachable(E.OfOperation.Schema, expr);
const codecs: [(schema: any, value: any) => unknown, (builders: any) => { Reachable(schema: any, text: any): unknown }][] = [
  [Plain.ToPlain.Reachable, Plain.FromPlain], [JSON.ToJSON.Reachable, JSON.FromJSON], [YAML.ToYAML.Reachable, YAML.FromYAML]];
for (const [encode, decode] of codecs) {
  const back = decode(E.Builders).Reachable(E.OfOperation.Schema, encode(E.OfOperation.Schema, expr));
  assert(back instanceof E.OfOperation.Data && back !== expr);
  assert(back.validate().length === 0 && JSON.ToJSON.Reachable(E.OfOperation.Schema, back) === expected);
}
const not_ = new E.OfOperation.Builder().name("not").arguments(true).create();
assert(JSON.ToJSON.Reachable(E.OfOperation.Schema, not_) ===
  '{"root": "s0", "objects": {"s0": {"kind": "operation", "name": "not", "arguments": [{"argument": {"$ref": "s1", "$schema": '
  + '"Expressions.OfLiteral"}, "index": 0}]}, "s1": {"kind": "literal", "bool": true}}}');
const alone = JSON.ToJSON.Reachable(E.OfLiteral.Schema, E.OfLiteral.resolve(-0));
assert(alone === '{"root": "s0", "objects": {"s0": {"kind": "literal", "float": -0.0}}}');
assert(Object.is((JSON.FromJSON(E.Builders).Reachable(E.OfLiteral.Schema, alone) as E.OfLiteral.Data).value, -0));
// nothing set: only the tag is written
assert(JSON.ToJSON.Reachable(E.OfLiteral.Schema, new E.OfLiteral.Data()) === '{"root": "s0", "objects": {"s0": {"kind": "literal"}}}');
assert(JSON.ToJSON.Reachable(E.OfOperation.Schema, new E.OfOperation.Data()) ===
  '{"root": "s0", "objects": {"s0": {"kind": "operation"}}}');
assert(JSON.ToJSON.Reachable(E.OfVariable.Schema, new E.OfVariable.Data()) === '{"root": "s0", "objects": {"s0": {"kind": "variable"}}}');
assert(JSON.ToJSON.Reachable(E.OfLet.Schema, new E.OfLet.Data()) === '{"root": "s0", "objects": {"s0": {"kind": "let"}}}');
raises(ValueError, () => JSON.FromJSON(E.Builders).Reachable(E.OfLiteral.Schema, '{"root": "s0", "objects": {"s0": {"kind": "operation"}}}'),
  "expected kind 'literal', got 'operation'");

## EXP-07 · Proxies build expressions too; the `used_by` entries they add are implied and ignored

In [ ]:
const proxy = JSON.FromJSON(Proxies.Builders).Reachable(E.OfOperation.Schema, expected);
const withInverse = JSON.ToJSON.Reachable(E.OfOperation.Schema, proxy as any);
assert(withInverse.includes('"used_by"') && withInverse !== expected);
const back = JSON.FromJSON(E.Builders).Reachable(E.OfOperation.Schema, withInverse);
assert(JSON.ToJSON.Reachable(E.OfOperation.Schema, back as any) === expected);

## EXP-08 · Validation and comparison work through the meta-schemas

In [ ]:
const validate = Validators.Validate(E.Builders);
assert(validate.Reachable(E.OfOperation.Schema, expr).length === 0 && validate(E.OfLiteral.Schema, lit).length === 0);
assert(same(validate(E.OfOperation.Schema, new E.OfOperation.Data(3n)), ["Expressions.OfOperation#0.name: expected str, got int"]));
raises(TypeError, () => JSON.ToJSON.Reachable(E.OfLiteral.Schema, new E.OfLiteral.Data([1n])), "a literal must hold a native value, got list");

function record(schema: S.OfObject.Data, expression: E.OfAny.Data): C.OfObject {
  return new C.OfObject(schema, expression);
}

const [one, two] = [E.OfLiteral.resolve(1n), E.OfLiteral.resolve(2n)];
assert(record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, E.OfLiteral.resolve(1n))) === 0);
assert(record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, two)) === null); // objects are not ordered
assert(record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, E.OfLiteral.resolve(1.0))) === null);
const f1 = E.OfOperation.resolve((o: E.OfOperation.Builder) => o.name("f").arguments(1n));
const f2 = E.OfOperation.resolve((o: E.OfOperation.Builder) => o.name("f").arguments(2n));
assert(record(E.OfOperation.Schema, f1).compare(record(E.OfOperation.Schema, f2)) === 0); // arguments are adjacencies

## EXP-09 · Builders read and write through the visitor protocols

In [ ]:
const seen: unknown[] = [];
const b = new E.OfLiteral.Builder();
b.properties((p) => seen.push(p.name()));
assert(same(seen, ["kind"]) && b.has("kind") && get_property(b, "kind") === "literal"); // the tag is always there
set_property(b, "kind", "literal");
assert(b.clear("kind") === b && b.has("kind"));
raises(ValueError, () => set_property(b, "kind", "let"), "expected kind 'literal', got 'let'");
seen.length = 0;
set_property(b, "float", 1.5);
assert(b.has("float") && !b.has("int") && get_property(b, "float") === 1.5);
set_property(b, "str", "s"); // another type replaces the value
assert(!b.has("float") && b.create().value === "s");
b.properties((p) => seen.push(`${p.name()}:${p.has()}`));
assert(same(seen, ["kind:true", "str:true"]));
assert(b.clear("int") === b && b.has("str")); // clearing another type changes nothing
assert(b.clear("str") === b && b.create().value === null);
raises(KeyError, () => b.property("nope", () => undefined), "unknown property 'nope'");
raises(TypeError, () => set_property(b, "int", true), "expected int, got bool");
raises(AttributeError, () => get_property(b, "int"), "property 'int' is not set");
b.property("int", (p) => p.value((a) => a.as_native((n) => n.set(1n).clear())).clear());
assert(!b.has("int"));
raises(TypeError, () => b.property("int", (p) => p.value((a) => a.as_object(() => undefined))), "property 'int' is native");
raises(TypeError, () => b.property("int", (p) => p.value((a) => a.as_union(() => undefined))), "property 'int' is native");
raises(TypeError, () => b.property("int", (p) => p.value((a) => a.as_intersection(() => undefined))), "property 'int' is native");
seen.length = 0;
assert(b.adjacencies((a) => seen.push(`${a.name()}:${a.me()}`)) === b && same(seen, ["used_by:argument"]));
raises(KeyError, () => b.adjacency("nope", () => undefined), "unknown adjacency 'nope'");
seen.length = 0;
b.adjacency("used_by", (a) => { a.add((e) => e.link("parent", (k) => k.set(op))); a.entries((e) => seen.push(e)); a.remove(null as any); });
assert(seen.length === 0); // used_by entries are ignored

const o = new E.OfOperation.Builder();
set_property(o, "name", "f");
assert(o.has("name") && !o.has("nope") && o.clear("nope") === o && get_property(o, "name") === "f");
seen.length = 0;
o.properties((p) => seen.push(p.name()));
assert(o.clear("name") === o && !o.has("name"));
o.properties((p) => seen.push(p.name()));
assert(same(seen, ["kind", "name", "kind"]));
raises(KeyError, () => o.property("arity", () => undefined), "unknown property 'arity'");
raises(KeyError, () => o.adjacency("nope", () => undefined), "unknown adjacency 'nope'");
seen.length = 0;
o.adjacencies((a) => seen.push(`${a.name()}:${a.me()}`));
assert(same(seen, ["arguments:parent", "used_by:argument"]));

/** Adds an `arguments` entry through the protocol, as FromPlain does. */
function add(builder: E.OfOperation.Builder, index: bigint | null, target: unknown): void {
  builder.adjacency("arguments", (a) => a.add((entry) => {
    if (target !== null) entry.link("argument", (k) => k.set(target as any));
    if (index !== null) set_property(entry, "index", index);
  }));
}

const [w, x, y, z] = ["w", "x", "y", "z"].map((v) => E.OfLiteral.resolve(v));
o.name("f");
add(o, 1n, y);
add(o, null, z);
add(o, 0n, x);
add(o, null, w);
assert(same(values(o.create()), ["x", "y", "z", "w"])); // by index; entries without one last, in order
seen.length = 0;

function read(entry: any): void {
  entry.links((k: any) => { seen.push(k.name()); k.target((t: unknown) => seen.push(t)); });
  entry.properties((p: any) => seen.push(p.name()));
  seen.push(entry.has("index") ? get_property(entry, "index") : null);
  seen.push(entry.has("index"));
  entry.clear("other");
  entry.clear("index");
  seen.push(entry.has("index"));
}

o.adjacency("arguments", (a) => a.entries((e) => { read(e); seen.push("|"); }));
assert(same(seen.slice(0, 7), ["argument", y, "index", 1n, true, false, "|"]));
raises(KeyError, () => o.adjacency("arguments", (a) => a.entries((e) => e.link("parent", () => undefined))), "'parent' is not a link this entry can set");
raises(KeyError, () => o.adjacency("arguments", (a) => a.entries((e) => e.property("nope", () => undefined))), "unknown property 'nope'");
o.adjacency("arguments", (a) => a.entries((e) => a.remove(e)));
assert(o.create().arguments.length === 0);
add(o, 0n, null);
raises(ValueError, () => o.create(), "link 'argument' is not set");
raises(ValueError, () => o.adjacency("arguments", (a) => a.entries((e) => e.link("argument", (k) => k.target((t) => seen.push(t))))), "link 'argument' is not set");
const fake = new E.OfOperation.Builder().name("f");
add(fake, 0n, new Fake("Contact"));
raises(TypeError, () => fake.create(), "an argument must be an expression, got Fake");

## EXP-10 · Variables and lets

In [ ]:
const v = new E.OfVariable.Builder().name("x").create();
assert(v.name === "x" && v.schema_name() === "Expressions.OfVariable" && E.OfVariable.resolve("x").name === "x");
assert(same(v.validate(), ["variable 'x' is not bound"]) && v.validate({ bound: ["x"] }).length === 0);
assert(same(new E.OfVariable.Data().validate(), ["a variable needs a name"]));
assert(same(new E.OfVariable.Data(1n).validate(), ["a variable's name must be a str, got int"]));
assert(E.OfVariable.resolve(v) === v && new E.OfVariable.Builder(v).name("y").clone().name === "y");
const let_ = new E.OfLet.Builder().name("x").value(1n).body(v).create();
assert(let_.name === "x" && (let_.value as E.OfLiteral.Data).value === 1n && let_.body === v && let_.validate().length === 0);
assert((new E.OfLet.Builder(let_).body(2n).clone().body as E.OfLiteral.Data).value === 2n && let_.body === v);
assert(new E.OfLet.Builder(let_).value(5n).update() === let_ && (let_.value as E.OfLiteral.Data).value === 5n && let_.body === v);
assert(E.OfLet.resolve((b: E.OfLet.Builder) => b.name("y").value(1n).body(2n)).name === "y" && E.OfLet.resolve(let_) === let_);
assert(same(new E.OfLet.Data("x").validate(), ["a let needs a value", "a let needs a body"]));
assert(same(new E.OfLet.Data(null, E.OfLiteral.resolve(1n), E.OfVariable.resolve("x")).validate(), [
  "a let needs a name", "body: variable 'x' is not bound"]));
const sameName = new E.OfLet.Data("y", E.OfVariable.resolve("y"), E.OfVariable.resolve("y"));
assert(same(sameName.validate(), ["value: variable 'y' is not bound"])); // the name is bound in the body only
assert((E.OfAny.resolve((a: E.OfAny.Builder) => a.as_variable("x")) as E.OfVariable.Data).name === "x");
assert(E.OfAny.resolve((a: E.OfAny.Builder) => a.as_let(let_)) === let_);
assert((new E.OfAny.Builder(v).clone() as E.OfVariable.Data).name === "x" && (new E.OfAny.Builder(let_).clone() as E.OfLet.Data).body === v);
const unnamed = [new E.OfOperation.Builder().create(), new E.OfVariable.Builder().create(), new E.OfLet.Builder().create()];
assert(same(unnamed.map((u) => u.name), [null, null, null]) && (unnamed[2] as E.OfLet.Data).value === null);
const openLet = new E.OfLet.Builder(new E.OfLet.Data("x")).clone();
assert(openLet.value === null && openLet.body === null);
raises(TypeError, () => E.OfLet.resolve(3n), "expected a let or a callable taking its builder, got 3");
raises(TypeError, () => E.OfVariable.resolve(3n), "expected a name, a variable or a callable taking its builder, got 3");
const vb = new E.OfVariable.Builder();
seen.length = 0;
assert(vb.adjacencies((a) => seen.push(a.name())) === vb && same(seen, ["used_by"])); // a variable has no arguments
raises(KeyError, () => vb.adjacency("arguments", () => undefined), "unknown adjacency 'arguments'");
const badLet = new E.OfLet.Builder().name("x");
add(badLet as any, 2n, E.OfLiteral.resolve(1n));
raises(ValueError, () => badLet.create(), "a let's value is argument 0 and its body argument 1, got index 2");

## EXP-11 · Terms write expressions with methods

In [ ]:
const self = E.variable("this");
const term = self.age.ge(18n);
assert(term.data instanceof E.OfOperation.Data && term.data.name === "ge" && (term.data.arguments[1] as E.OfLiteral.Data).value === 18n);
const got = (term.data as E.OfOperation.Data).arguments[0] as E.OfOperation.Data;
assert(got.name === "get" && got.arguments[0] === self.data && (got.arguments[1] as E.OfLiteral.Data).value === "age");
assert(((self.get("eq").data as E.OfOperation.Data).arguments[1] as E.OfLiteral.Data).value === "eq"); // a property named like a method
const methods = [self.eq(1n), self.ne(1n), self.lt(1n), self.le(1n), self.gt(1n), self.ge(1n), self.and_(true), self.or_(true),
  self.not_(), self.implies(true), self.add(1n), self.sub(1n), self.mul(1n), self.neg(), self.has("x"), self.get("x")];
assert(same(methods.map((m) => (m.data as E.OfOperation.Data).name), ["eq", "ne", "lt", "le", "gt", "ge", "and", "or", "not",
  "implies", "add", "sub", "mul", "neg", "has", "get"]));
assert((E.literal(3n).data as E.OfLiteral.Data).value === 3n && (E.operation("in", 1n, 2n).data as E.OfOperation.Data).name === "in");
const bound = E.let_("a", 1n, E.variable("a"));
assert((bound.data as E.OfLet.Data).name === "a" && E.OfLet.resolve(bound) === bound.data && E.OfOperation.resolve(term) === term.data);
assert(E.OfAny.resolve(term) === term.data && E.OfAny.resolve((a: E.OfAny.Builder) => a.as_operation(term)) === term.data);
raises(TypeError, () => E.OfOperation.resolve(self), "expected an operation or a callable taking its builder"); // a variable, not an operation
assert(self.then === undefined && self.toJSON === undefined && self[Symbol.iterator] === undefined); // JavaScript's own probes are not properties